In [ ]:
pip install pdfplumber

In [ ]:
#checking if it is properly able to extract the file contents
import pdfplumber

pdf_path = "pdf_invoice_files"
with pdfplumber.open(pdf_path) as pdf:
    full_text=""
    for page in pdf.pages:
        full_text+= page.extract_text() + "\n"

In [ ]:
import pdfplumber
import os
import re
import pandas as pd
from dateutil import parser

def right_date(text):
    try:
        
        date_obj = parser.parse(text, fuzzy=True)
        return date_obj.strftime("%Y-%m-%d")
    except (ValueError, TypeError):
        return "Manual Check Required"

def smart_parse(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        # Combine all pages into one string of text
        full_text = ""
        for page in pdf.pages:
            full_text += page.extract_text() + "\n"

        amounts = re.findall(r"(?:Total\s*Due|Total|Amount|Balance)\s*[:\s]*[^\d]*([\d,]+\.\d{2})", full_text, re.IGNORECASE)
        final_amount = amounts[-1] if amounts else 0.0
        
        date_match = re.search(r"(?:Invoice\s*Date|Date|Dated)\s*[:\s]*([^\n]+)", full_text, re.IGNORECASE)
        if date_match:
            potential_date= date_match.group(1).strip()
            potential_date1 = re.split(r's{2,}', potential_date)
            potential_date = potential_date1[0]
            formatted_date= right_date(potential_date)
        else:
            formatted_date = "Not Found

        inv_pattern = r"(?i)(?:Invoice\s*Number|Invoice|Inv|Bill\s*No)[\s:#]*([A-Z0-9]+(?:-[A-Z0-9]+)+|[A-Z]{2,}\d+)"
        inv_match = re.search(inv_pattern, full_text)
        if inv_match:
            invoice_no = inv_match.group(1).strip()
        else:
            secondary_pattern = r"(?i)Invoice[\s:#]*([A-Z0-9-]+)"
            secondary_match = re.search(secondary_pattern, full_text)
            invoice_no = secondary_match.group(1).strip() if secondary_match else "Not Found"

        return {
            "file_name": os.path.basename(pdf_path),
            "invoice_no": invoice_no if invoice_no else "Not Found",
            "date": formatted_date if date_match else "Not Found",
            "amount": float(final_amount.replace(',', '')) if isinstance(final_amount, str) else final_amount
        }

# --- Execution ---
folder_path = "pdf_invoice_files"
files = [f"{folder_path}/{f}" for f in os.listdir(folder_path) if f.endswith(".pdf")]
results = []

for f in files:
    data = smart_parse(f)
    results.append(data)

# Convert to a DataFrame to see your hard work!
df = pd.DataFrame(results)
print(df)
            

In [ ]:
total_spending = df['amount'].sum()

invoice_count = len(df)

average_spending = df['amount'].mean()

print("\033[1m" + "Daily Operational Report" + "\033[0m")

print(f"Total Invoiced Processed: {invoice_count}")

print(f"Total Amount Parsed: {total_spending:,.2f}")

print(f"Average Invoice Value: {average_spending:,.2f}")

In [ ]:
#Creating the database in SQl
import sqlite3

conn = sqlite3.connect("Invoice_Records.db")

df.to_sql("Processed_Invoices", conn, if_exists = "replace", index= False)

print("Database update complete.")
print("Data has been successfully saved to Invoice_Records.db")

In [ ]:
%pip install ipython-sql sqlalchemy

In [ ]:
%reload_ext sql
%sql sqlite:///Invoice_Records.db

In [ ]:
#resolve compatibility issues between the ipython-sql extension and newer versions of the prettytable library.
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [ ]:
%sql select * from Processed_Invoices;

In [ ]:
%%sql
select sum(amount) as Total_Business_Spending
from Processed_Invoices;

In [ ]:
%%sql
select strftime('%Y-%m', date) as Month,
count(invoice_no) as Invoice_count,
sum(amount) as Monthly_spending
from Processed_Invoices
group by Month
order by Month DESC;

In [ ]:
import matplotlib.pyplot as plt

df_plot = pd.read_sql("select date, amount from Processed_Invoices order by date", conn)

plt.figure(figsize=(10,5))
plt.bar(df_plot['date'], df_plot['amount'], color = 'skyblue')
plt.title("Invoice spend over time")
plt.xlabel('Date')
plt.ylabel('Amount (₹)')
#plt.xticks(rotation=45)
plt.show()